# EarlyStruct – Design Explorer (Floors)
Carbon-first feasibility & comparison tool.

**How to use**
1) Set **Data dir** to the folder containing your CSVs.
2) (Optional) Provide a **Control file** path. If present, it takes precedence for project details and SPANS unless it contains `USE_CSV = Y`.
3) (Optional) Enter **Spans** like `9, 10.5` (meters) or `28ft, 32ft`. Notebook spans override control-file SPANS.
4) Click **Run Evaluation**. If no spans are provided (and no grid_options.csv / ideal spacing), the tool runs a 1-ft sweep (18–45 ft).


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from earlystruct import cli
from earlystruct.core import reporting

# ensure inline plotting in notebook and a consistent backend
%matplotlib inline

import importlib, matplotlib
# optional: enforce the inline backend if your environment is picky
matplotlib.use("module://ipykernel.pylab.backend_inline")

In [ ]:
data_dir = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/csvs")
control_file = "/Users/benjaminsalop/Desktop/Oxford/Research/edca/control_file/control_file.txt"

df, ranked, pareto, saved = cli.evaluate(
    data_dir=data_dir,
    spans_str=None,          # let control file + sweep logic handle it
    export_dir=None,
    control_file=control_file,
)

if isinstance(saved, dict):
    project = saved.get("project", {}) or {}
    ctl = saved.get("ctl", {}) or {}
else:
    project = {}
    ctl = {}

df.shape, df.columns

In [ ]:
import os, sys
from pathlib import Path

CODE_ROOT = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/code")
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from earlystruct.core.control import parse_control

control_path = "/Users/benjaminsalop/Desktop/Oxford/Research/edca/control_file/control_file.txt"

print("Exists?", os.path.exists(control_path), "->", control_path)

ctl = parse_control(control_path)
print("Parsed keys:", list(ctl.keys()))

print("PROJECT_NAME:", ctl.get("PROJECT_NAME"))
print("UNIT:", ctl.get("UNIT"))
print("FLOOR_AREA_PER_FLOOR:", ctl.get("FLOOR_AREA_PER_FLOOR"))
print("SPAN_SWEEP_FROM_MIN:", ctl.get("SPAN_SWEEP_FROM_MIN"), "BOOL:", ctl.get("SPAN_SWEEP_FROM_MIN_BOOL"))
print("ONE_WAY_IRREGULAR:", ctl.get("ONE_WAY_IRREGULAR"), "BOOL:", ctl.get("ONE_WAY_IRREGULAR_BOOL"))
print("ONE_WAY_SLAB_MIN_SPAN:", ctl.get("ONE_WAY_SLAB_MIN_SPAN"))
print("ONE_WAY_BEAM_MIN_SPAN:", ctl.get("ONE_WAY_BEAM_MIN_SPAN"))
print("PROGRAM_BLOCKS:", ctl.get("PROGRAM_BLOCKS"))


In [ ]:
UNIT = ctl.get("UNIT", "metric").lower()
is_imperial = UNIT.startswith("imp") or ("ft" in UNIT)

FT_PER_M = 3.28084

df_disp = df.copy()

if is_imperial:
    df_disp["span_slab_dir_ft"] = df_disp["span_slab_dir_m"] * FT_PER_M
    df_disp["span_beam_dir_ft"] = df_disp["span_beam_dir_m"] * FT_PER_M
else:
    df_disp["span_slab_dir_ft"] = pd.NA
    df_disp["span_beam_dir_ft"] = pd.NA

cols = [
    "system_id", "type", "feasible",
    "span_slab_dir_m", "span_beam_dir_m",
    "span_slab_dir_ft", "span_beam_dir_ft",
    "depth_m", "area_m2",
    "concrete_m3", "steel_m3", "timber_m3",
    "cost_per_m2", "carbon_per_m2",
    "cost_total", "carbon_total_kg",
]

df_disp[[c for c in cols if c in df_disp.columns]].head(15)


In [ ]:
best_by_type = reporting.cheapest_span_by_type(df)

best_cols = [
    "type",
    "system_id",
    "span_slab_dir_m",
    "span_beam_dir_m",
    "depth_m",
    "area_m2",
    "concrete_m3", "steel_m3", "timber_m3",
    "cost_total", "cost_per_m2",
    "carbon_total_kg", "carbon_per_m2",
]

best_by_type[[c for c in best_cols if c in best_by_type.columns]]


In [ ]:
reporting.print_verbose_summary(df, project, ctl)


In [ ]:
def prepare_for_plots(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()

    # Ensure feasible is boolean
    if "feasible" in df.columns:
        if df["feasible"].dtype == object:
            df["feasible"] = df["feasible"].astype(str).str.strip().str.upper().map(
                {"Y": True, "YES": True, "TRUE": True, "T": True, "1": True,
                 "N": False, "NO": False, "FALSE": False, "0": False}
            ).fillna(False)
        else:
            try:
                df["feasible"] = df["feasible"].astype(bool)
            except Exception:
                df["feasible"] = df["feasible"].fillna(False).astype(bool)
    else:
        df["feasible"] = False

    # numeric coercion for columns used in plots
    for col in ("carbon_per_m2", "cost_per_m2", "cost_total", "carbon_total_kg"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # typology convenience (if not present)
    if "typology" not in df.columns:
        df = reporting.add_typology(df)

    # ensure type and system_id are string (avoid mixed-type sort errors)
    if "type" in df.columns:
        df["type"] = df["type"].astype(str)
    if "system_id" in df.columns:
        df["system_id"] = df["system_id"].astype(str)

    return df

df_clean = prepare_for_plots(df)
print("Prepared df_clean — rows:", len(df_clean))


In [ ]:
# Use df_clean that you prepared earlier.
# After each plot call, capture the current figure and save it immediately.

plots_dir = Path(".") / "plots"
plots_dir.mkdir(exist_ok=True)

# Pareto
reporting.plot_pareto(df_clean, title="Carbon vs Cost (Pareto)")
plt.gcf().savefig(plots_dir / "pareto.png", bbox_inches="tight")
print("Saved:", plots_dir / "pareto.png")

# Best by typology (carbon)
reporting.plot_best_typology_carbon(df_clean, title="Lowest-carbon option by typology")
plt.gcf().savefig(plots_dir / "best_by_typology.png", bbox_inches="tight")
print("Saved:", plots_dir / "best_by_typology.png")

# Best by type (carbon)
reporting.plot_best_type_carbon(df_clean, title="Lowest-carbon option by type")
plt.gcf().savefig(plots_dir / "best_by_type.png", bbox_inches="tight")
print("Saved:", plots_dir / "best_by_type.png")

In [ ]:
# Choose which DF to plot: df (raw), or df_clean if you prepared it earlier.
plot_df = df  # or df_clean if you prefer the cleaned version

# Minimum span threshold (None or numeric)
min_span_m = 0.0   # change to e.g. 6.6 to show only >= 6.6 m spans

# Call plot helper
reporting.plot_carbon_vs_span(
    plot_df,
    span_col="span_slab_dir_m",
    carbon_col="carbon_per_m2",
    min_span=min_span_m,
    only_feasible=True,
    title="Embodied carbon vs span (colored by typology)"
)

# Save to disk: capture current figure
out_dir = Path(".") / "plots"
out_dir.mkdir(exist_ok=True)
import matplotlib.pyplot as plt
plt.gcf().savefig(out_dir / "carbon_vs_span.png", bbox_inches="tight")
print("Saved:", out_dir / "carbon_vs_span.png")

reporting.plot_carbon_vs_span_grid(
    plot_df,
    span_min_global=6.6,
    span_max_global=10.0,   # set high enough to include all desired spans, or None to auto-infer
    step=0.25,              # quarter-meter resolution (dense)
    carbon_col="carbon_per_m2",
    only_feasible=False,    # set False to include all numeric rows regardless of feasible flag
    max_points=200000,
    title="Embodied carbon vs span — full expanded grid (>=6.6 m)"
)

# Plot the full expanded grid, colored by `type` (use only_feasible=False to show whole catalog)
reporting.plot_carbon_vs_span_grid_types(df, span_min_global=6.6, span_max_global=10.0, step=0.25, only_feasible=False)


# Save
out_dir = Path(".") / "plots"
out_dir.mkdir(exist_ok=True)
import matplotlib.pyplot as plt
plt.gcf().savefig(out_dir / "carbon_vs_span_expanded.png", bbox_inches="tight")
print("Saved:", out_dir / "carbon_vs_span_expanded.png")


In [ ]:
# Diagnostic: inspect span/min/max and carbon columns
import pandas as pd, numpy as np
from pathlib import Path

# Use df (the evaluated results) or the raw systems catalog if you prefer
print("rows in df:", len(df))

cols_to_check = [
    "system_id", "type", "typology", "category",
    "span_slab_dir_m", "span_beam_dir_m", "span_input_m", "span_m",
    "max_span", "max_span_m", "span_max",
    "carbon_per_m2", "carbon_total_kg", "cost_per_m2"
]

present = [c for c in cols_to_check if c in df.columns]
print("present columns checked:", present)

# Show dtypes and basic stats
print("\nDTYPES and non-null counts:")
print(df[present].dtypes)
print("\ncarbon_per_m2 stats (coerced to numeric):")
cnum = pd.to_numeric(df.get("carbon_per_m2"), errors="coerce")
print("  non-null:", cnum.notna().sum(), "min/median/max:", cnum.min(), cnum.median(), cnum.max())

# Show rows with extreme carbon values
print("\nRows with carbon_per_m2 > 5000 or < 0 (possible bad values):")
display(df.loc[(cnum > 5000) | (cnum < 0), present].head(20))

# Build row_min and row_max used by plotting (same logic as plotting)
def infer_row_min(row):
    for c in ("span_slab_dir_m","span_input_m","span_m","span_x_m","span_y_m"):
        if c in row.index:
            v = pd.to_numeric(row.get(c), errors="coerce")
            if not pd.isna(v):
                return v
    return float("nan")

def infer_row_max(row):
    for c in ("max_span","max_span_m","span_max"):
        if c in row.index:
            v = pd.to_numeric(row.get(c), errors="coerce")
            if not pd.isna(v):
                return v
    return float("nan")

sample = df.head(40).copy()
sample["_row_min"] = sample.apply(infer_row_min, axis=1)
sample["_row_max"] = sample.apply(infer_row_max, axis=1)
sample["_carbon_numeric"] = pd.to_numeric(sample.get("carbon_per_m2"), errors="coerce")
print("\nSample first 40 rows with inferred min/max/carbon:")
display(sample[["system_id","type","_row_min","_row_max","_carbon_numeric"]].head(40))


In [ ]:
# Cell A: simple scatter using df_disp
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# df_disp should already exist in your notebook (from earlier)
if "df_disp" not in globals():
    print("df_disp not found — make sure you ran the earlier cells that create it (df_disp = df.copy() etc.)")
else:
    D = df_disp.copy()

    # Ensure columns exist
    if "span_slab_dir_m" not in D.columns:
        raise RuntimeError("Column 'span_slab_dir_m' not found in df_disp.")
    if "carbon_per_m2" not in D.columns:
        raise RuntimeError("Column 'carbon_per_m2' not found in df_disp.")
    if "type" not in D.columns:
        D["type"] = D.get("typology", "Unknown")

    # coerce numeric
    D["_span"] = pd.to_numeric(D["span_slab_dir_m"], errors="coerce")
    D["_carbon"] = pd.to_numeric(D["carbon_per_m2"], errors="coerce")

    # drop rows missing numeric span or carbon
    plot_df = D.loc[D["_span"].notna() & D["_carbon"].notna()].copy()
    print("Rows plotted (df_disp scatter):", len(plot_df))

    # color by 'type'
    types = list(pd.unique(plot_df["type"].astype(str)))
    colors = plt.rcParams.get("axes.prop_cycle").by_key().get("color", None)
    if not colors:
        colors = ["C%d" % i for i in range(10)]
    color_map = {t: colors[i % len(colors)] for i, t in enumerate(types)}

    fig, ax = plt.subplots(figsize=(11,6))
    for t in types:
        sub = plot_df[plot_df["type"].astype(str) == t]
        if sub.empty:
            continue
        ax.scatter(sub["_span"], sub["_carbon"], s=28, alpha=0.8, label=str(t), c=color_map.get(t), edgecolors="none")

    ax.set_xlabel("Span (slab dir) — span_slab_dir_m (m)")
    ax.set_ylabel("Embodied carbon (kgCO₂e / m²) — carbon_per_m2")
    ax.set_title("Carbon vs slab span (df_disp) — colored by type")
    ax.grid(True, linestyle=":", alpha=0.4)

    # Legend handling: if many types, place legend outside
    if len(types) <= 20:
        ax.legend(title="Type", loc="best")
    else:
        ax.legend(title="Type", bbox_to_anchor=(1.02,1), loc="upper left", ncol=1)
        plt.subplots_adjust(right=0.75)

    plt.tight_layout()
    plt.show()


In [ ]:
# Cell B: catalog-based scatter using systems_catalog.max_span and df results
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

# adjust path if your CSV is located elsewhere
catalog_csv = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/csvs/systems_catalog.csv")   # user-provided path seen earlier
if not catalog_csv.exists():
    print("systems_catalog.csv not found at", catalog_csv)
else:
    systems = pd.read_csv(catalog_csv)

    # sanity check columns
    if "system_id" not in systems.columns:
        raise RuntimeError("systems_catalog.csv missing 'system_id' column.")
    if "max_span" not in systems.columns:
        raise RuntimeError("systems_catalog.csv missing 'max_span' column (this is required per your instruction).")

    # Build best-observed carbon per system from df (use df if present, else try df_disp)
    df_source = globals().get("df", None) or globals().get("df_disp", None)
    if df_source is None:
        raise RuntimeError("No df or df_disp available in notebook. Run the evaluation cell first to produce df.")

    # ensure system_id in df_source
    if "system_id" not in df_source.columns:
        # maybe system id column is named differently — try lower-case fallback
        if "system_id" in (c.lower() for c in df_source.columns):
            # attempt to align by case-insensitive match
            for c in df_source.columns:
                if c.lower() == "system_id":
                    df_source = df_source.rename(columns={c: "system_id"})
                    break
        else:
            raise RuntimeError("df does not contain a system_id column — cannot map results to catalog.")

    # coerce numeric carbon
    df_source["_carbon_num"] = pd.to_numeric(df_source.get("carbon_per_m2"), errors="coerce")

    # For each system_id pick the minimum carbon_per_m2 observed across rows (you can change to mean/min/median)
    best_by_system = (
        df_source.loc[df_source["_carbon_num"].notna()]
                 .groupby("system_id", as_index=True)["_carbon_num"]
                 .min()
                 .rename("carbon_best_per_m2")
                 .reset_index()
    )

    # merge with systems_catalog on system_id
    merged = systems.merge(best_by_system, how="left", on="system_id")

    # If carbon_best_per_m2 is missing, try to fall back to df overall average
    fallback_carbon = df_source["_carbon_num"].mean() if df_source["_carbon_num"].notna().any() else np.nan
    merged["carbon_best_per_m2"] = merged["carbon_best_per_m2"].fillna(fallback_carbon)

    # coerce numeric span
    merged["_max_span"] = pd.to_numeric(merged["max_span"], errors="coerce")
    plot_df = merged.loc[merged["_max_span"].notna() & merged["carbon_best_per_m2"].notna()].copy()
    print("Rows plotted (catalog scatter):", len(plot_df))

    # color by type column in systems_catalog (prefer that)
    type_col = "type" if "type" in merged.columns else None
    if type_col is None:
        merged["type"] = merged.get("category", "Unknown")
        type_col = "type"

    types = list(pd.unique(plot_df[type_col].astype(str)))
    colors = plt.rcParams.get("axes.prop_cycle").by_key().get("color", None)
    if not colors:
        colors = ["C%d" % i for i in range(10)]
    color_map = {t: colors[i % len(colors)] for i, t in enumerate(types)}

    fig, ax = plt.subplots(figsize=(11,6))
    for t in types:
        sub = plot_df[plot_df[type_col].astype(str) == t]
        if sub.empty:
            continue
        ax.scatter(sub["_max_span"], sub["carbon_best_per_m2"], s=40, alpha=0.85, label=str(t), c=color_map.get(t), edgecolors="none")

    ax.set_xlabel("Catalog max_span (m) — systems_catalog.max_span")
    ax.set_ylabel("Best observed carbon (kgCO₂e / m²) — from df results")
    ax.set_title("Catalog max_span vs best-observed carbon_per_m2 (by system), colored by type")
    ax.grid(True, linestyle=":", alpha=0.4)

    if len(types) <= 20:
        ax.legend(title="Type", loc="best")
    else:
        ax.legend(title="Type", bbox_to_anchor=(1.02,1), loc="upper left")
        plt.subplots_adjust(right=0.75)

    plt.tight_layout()
    plt.show()


In [ ]:
# Corrected Cell B: catalog-based scatter using systems_catalog.max_span and df results
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

catalog_csv = Path("/Users/benjaminsalop/Desktop/Oxford/Research/edca/csvs/systems_catalog.csv")
if not catalog_csv.exists():
    raise RuntimeError("systems_catalog.csv not found at: " + str(catalog_csv))

systems = pd.read_csv(catalog_csv)

# sanity checks
if "system_id" not in systems.columns:
    raise RuntimeError("systems_catalog.csv missing 'system_id' column.")
if "max_span" not in systems.columns:
    raise RuntimeError("systems_catalog.csv missing 'max_span' column (required).")

# Get df source safely from globals
if "df" in globals() and isinstance(globals()["df"], pd.DataFrame):
    df_source = globals()["df"]
elif "df_disp" in globals() and isinstance(globals()["df_disp"], pd.DataFrame):
    df_source = globals()["df_disp"]
else:
    raise RuntimeError("No df or df_disp DataFrame found in notebook. Run the evaluation cell first to produce df.")

# Ensure system_id in df_source (case-insensitive fallback)
if "system_id" not in df_source.columns:
    lower_map = {c.lower(): c for c in df_source.columns}
    if "system_id" in lower_map:
        df_source = df_source.rename(columns={lower_map["system_id"]: "system_id"})
    else:
        raise RuntimeError("df does not contain a 'system_id' column — cannot map results to systems_catalog.")

# Coerce numeric carbon
df_source = df_source.copy()
df_source["_carbon_num"] = pd.to_numeric(df_source.get("carbon_per_m2"), errors="coerce")

# Build best-observed carbon per system (minimum across rows)
best_by_system = (
    df_source.loc[df_source["_carbon_num"].notna()]
             .groupby("system_id", as_index=True)["_carbon_num"]
             .min()
             .rename("carbon_best_per_m2")
             .reset_index()
)

# Merge with catalog on system_id
merged = systems.merge(best_by_system, how="left", on="system_id")

# Fallback carbon if missing (use overall mean of df_source if available)
fallback_carbon = df_source["_carbon_num"].mean() if df_source["_carbon_num"].notna().any() else np.nan
merged["carbon_best_per_m2"] = merged["carbon_best_per_m2"].fillna(fallback_carbon)

# coerce numeric span
merged["_max_span"] = pd.to_numeric(merged["max_span"], errors="coerce")
plot_df = merged.loc[merged["_max_span"].notna() & merged["carbon_best_per_m2"].notna()].copy()
print("Rows plotted (catalog scatter):", len(plot_df))

# Use 'type' column from systems_catalog if present, else fall back to 'category'
type_col = "type" if "type" in systems.columns else None
if type_col is None:
    merged["type"] = merged.get("category", "Unknown")
    type_col = "type"

# Build color mapping
types = list(pd.unique(plot_df[type_col].astype(str)))
colors = plt.rcParams.get("axes.prop_cycle").by_key().get("color", None)
if not colors:
    colors = ["C%d" % i for i in range(10)]
color_map = {t: colors[i % len(colors)] for i, t in enumerate(types)}

# Plot
fig, ax = plt.subplots(figsize=(11,6))
for t in types:
    sub = plot_df[plot_df[type_col].astype(str) == t]
    if sub.empty:
        continue
    ax.scatter(sub["_max_span"], sub["carbon_best_per_m2"], s=40, alpha=0.85, label=str(t),
               c=color_map.get(t), edgecolors="none")

ax.set_xlabel("Catalog max_span (m) — systems_catalog.max_span")
ax.set_ylabel("Best observed carbon (kgCO₂e / m²) — from df results")
ax.set_title("Catalog max_span vs best-observed carbon_per_m2 (by system), colored by type")
ax.grid(True, linestyle=":", alpha=0.4)

if len(types) <= 20:
    ax.legend(title="Type", loc="best")
else:
    ax.legend(title="Type", bbox_to_anchor=(1.02,1), loc="upper left")
    plt.subplots_adjust(right=0.75)

plt.tight_layout()
plt.show()
